# churn_predict - rebuild the generation-B artifacts on Kaggle

The live Streamlit app must publish the figures the CV and README quote: Voting ROC-AUC
0.898, precision@10% 0.487, MILP $202,403 / 20.24x / 400 customers at a $10,000 budget.

Those come from commit `ff884b3` (24 features, 2022-03-19 split, fixed depth-5 tree,
GradientBoosting scores). This notebook clones that commit, retrains the suite with the
pinned environment, checks the figures reproduce, and writes a pruned runtime artifact.

Output: `/kaggle/working/artifacts_slim.pkl` -> commit as `models/artifacts_slim.pkl`.

In [ ]:
# Only the modelling stack is pinned. numpy/pandas/scipy stay as Kaggle ships them:
# forcing numpy==2.4.6 over the image's build breaks numpy._core imports.
!pip install -q "scikit-learn==1.9.1" "scikit-survival==0.28.0" "xgboost==3.2.*" "shap==0.51.*"
import sklearn, numpy, pandas, scipy, sksurv, xgboost, shap
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| sksurv", sksurv.__version__,
      "| xgboost", xgboost.__version__, "| shap", shap.__version__)

In [ ]:
import os, sys, subprocess, time, urllib.request, tarfile

COMMIT = "ff884b3"
URL = f"https://github.com/bravo2024/churn_predict/archive/{COMMIT}.tar.gz"
os.makedirs("/kaggle/working/src", exist_ok=True)
tgz = "/kaggle/working/src/repo.tar.gz"
urllib.request.urlretrieve(URL, tgz)

with tarfile.open(tgz) as t:
    root = t.getnames()[0].split("/")[0]
    t.extractall("/kaggle/working/src")

REPO = os.path.join("/kaggle/working/src", root)
print("repo:", REPO)

TM = os.path.join(REPO, "train_models.py")
src = open(TM, encoding="utf-8").read()
assert '"Hispanic"' in src and "max_depth=5" in src and "2022-03-19" in src, \
    "not the generation-B code"

# commit ff884b3 scores the test period with GradientBoosting only, while the notebook MILP
# (which the CV and README quote: $202,403 / 20.24x) uses the Voting ensemble. Later commits
# carry the fix, so patch only when the old line is still present.
GB_LINE = 'prob_te = fitted["GradientBoosting (Ch7)"].predict_proba(Xp_te)[:, 1]'
VOTE_LINE = 'prob_te = fitted["Voting (Ch7)"].predict_proba(Xp_te)[:, 1]  # matches the notebook MILP'
if GB_LINE in src:
    open(TM, "w", encoding="utf-8").write(src.replace(GB_LINE, VOTE_LINE))
    print("patched test-period scoring to the Voting ensemble")
else:
    assert 'fitted["Voting (Ch7)"].predict_proba(Xp_te)' in src, \
        "unexpected test-period scoring line"
    print("source already scores with the Voting ensemble")

t0 = time.time()
res = subprocess.run([sys.executable, "train_models.py"], cwd=REPO,
                     capture_output=True, text=True)
print(res.stdout[-4000:])
print(res.stderr[-2000:])
print(f"train_models.py exit={res.returncode} in {time.time()-t0:.0f}s")
assert res.returncode == 0, "retrain failed"

In [ ]:
import pickle, os, hashlib, warnings
warnings.filterwarnings("ignore")

FULL = os.path.join(REPO, "models", "artifacts.pkl")
print(f"full artifact: {os.path.getsize(FULL)/1e6:.1f} MB")
A = pickle.load(open(FULL, "rb"))

m = A["churn_metrics_df"]
print("\n" + m.to_string())
v = m.loc["Voting (Ch7)"]
milp = A["offer_result"]["milp_by_budget"][10000]
print(f"\nVoting  AUC={v['ROC-AUC']:.4f}  Prec@10%={v['Prec@10%']:.4f}")
print(f"MILP    rev=${milp['rev']:,.0f}  roas={milp['roas']:.2f}x  customers={milp['customers']}")
print("summary:", A["churn_summary"])
print("sensitivity:\n", A["sensitivity_df"].to_string(index=False))

# The published figures. Tolerance is tight: anything outside means the rebuild drifted
# and the commit must be regenerated rather than shipped.
assert abs(v["ROC-AUC"] - 0.8976) < 0.002, f"AUC drifted: {v['ROC-AUC']:.4f}"
assert abs(v["Prec@10%"] - 0.4866) < 0.005, f"precision drifted: {v['Prec@10%']:.4f}"
assert abs(milp["rev"] - 202403) < 600, f"MILP drifted: ${milp['rev']:,.0f}"
assert abs(milp["roas"] - 20.24) < 0.10, f"ROAS drifted: {milp['roas']:.2f}"
assert milp["customers"] == 400, f"offer count drifted: {milp['customers']}"
print("\nOK - reproduces the CV / README figures")

DROP = ("tr", "te", "pre", "pre_s", "pca6", "kmeans", "prob_te",
        "segment_profiles", "seg_names", "silhouette")
slim = {k: val for k, val in A.items() if k not in DROP}
slim["survival"] = {k: val for k, val in A["survival"].items() if k not in ("rsf", "cox")}

OUT = "/kaggle/working/artifacts_slim.pkl"
pickle.dump(slim, open(OUT, "wb"), protocol=4)
print(f"\nslim artifact: {os.path.getsize(OUT)/1e6:.1f} MB  "
      f"({os.path.getsize(OUT)/os.path.getsize(FULL)*100:.1f}% of full)")
print("sha256:", hashlib.sha256(open(OUT, "rb").read()).hexdigest())
print("keys:", sorted(slim.keys()))

READS = ["churn_metrics_df", "churn_summary", "cust", "feature_names", "fitted_models",
         "offer_result", "perm_imp", "rf_imp", "sensitivity_df", "shap_importance",
         "survival", "test_scores"]
assert not [k for k in READS if k not in slim], "missing dashboard keys"
assert set(slim["survival"]) == {"cindex_cox", "cindex_rsf", "corr_s90_binary", "horizons", "probs"}
print("contract OK - 12 keys, survival", sorted(slim["survival"]))

# copy the regenerated CSVs so the repo data matches the artifact
for f in ("churn_scores_test.csv", "offer_allocation.csv"):
    src = os.path.join(REPO, "models", f)
    dst = os.path.join("/kaggle/working", f)
    open(dst, "wb").write(open(src, "rb").read())
    print(f"copied {f}: {os.path.getsize(dst)} bytes")